# Explore here

In [ ]:
# Your code here
import os
import shutil

ruta_carpeta = '../data/raw/dogs-vs-cats/train/'
for filename in os.listdir(ruta_carpeta):
    if filename.startswith('cat'):
            destino = os.path.join('../data/raw/', 'cat')
            shutil.move(os.path.join(ruta_carpeta, filename), os.path.join(destino, filename))
            print(f'Movido {filename} a la carpeta cat')
    elif filename.startswith('dog'):
            destino = os.path.join('../data/raw/', 'dog')
            shutil.move(os.path.join(ruta_carpeta, filename), os.path.join(destino, filename))
            print(f'Movido {filename} a la carpeta dog')

In [6]:
import random

ruta_base = '../data/raw/'

clases = ['cat', 'dog']
ratio_train_test = 0.8 #80% para entrenamiento, 20% para prueba

#Semilla para reproducibilidad
random.seed(42)

ruta_dataset = os.path.join(ruta_base, 'dataset')

for clase in clases:
    #armo la ruta de la clase
    ruta_clase = os.path.join(ruta_base, clase)
    #listo las imágenes de la clase
    imagenes = [archivo for archivo in os.listdir(ruta_clase)]
    #las mezclo aleatoriamente
    random.shuffle(imagenes)

    #calculo el número de imágenes para entrenamiento y prueba
    num_train = int(len(imagenes) * ratio_train_test)
    #dejo la lista de imágenes de entrenamiento y prueba
    imagenes_train = imagenes[:num_train]
    imagenes_test = imagenes[num_train:]

    #mover a train
    for imagen in imagenes_train:
        ruta_origen = os.path.join(ruta_clase, imagen)
        ruta_destino = os.path.join(ruta_dataset, 'train', clase, imagen)
        shutil.move(ruta_origen, ruta_destino)
        print(f'Movido {imagen} a la carpeta train/{clase}')

    #mover a test
    for imagen in imagenes_test:
        ruta_origen = os.path.join(ruta_clase, imagen)
        ruta_destino = os.path.join(ruta_dataset, 'test', imagen)
        shutil.move(ruta_origen, ruta_destino)
        print(f'Movido {imagen} a la carpeta test/{clase}')


Movido cat.3358.jpg a la carpeta train/cat
Movido cat.8711.jpg a la carpeta train/cat
Movido cat.11206.jpg a la carpeta train/cat
Movido cat.867.jpg a la carpeta train/cat
Movido cat.4128.jpg a la carpeta train/cat
Movido cat.11904.jpg a la carpeta train/cat
Movido cat.9351.jpg a la carpeta train/cat
Movido cat.8371.jpg a la carpeta train/cat
Movido cat.1364.jpg a la carpeta train/cat
Movido cat.5447.jpg a la carpeta train/cat
Movido cat.8520.jpg a la carpeta train/cat
Movido cat.6905.jpg a la carpeta train/cat
Movido cat.2063.jpg a la carpeta train/cat
Movido cat.11724.jpg a la carpeta train/cat
Movido cat.3263.jpg a la carpeta train/cat
Movido cat.107.jpg a la carpeta train/cat
Movido cat.11471.jpg a la carpeta train/cat
Movido cat.7355.jpg a la carpeta train/cat
Movido cat.9395.jpg a la carpeta train/cat
Movido cat.11709.jpg a la carpeta train/cat
Movido cat.2053.jpg a la carpeta train/cat
Movido cat.2331.jpg a la carpeta train/cat
Movido cat.8631.jpg a la carpeta train/cat
Movido c

In [28]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Definir los generadores de datos para entrenamiento y prueba
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Crear los generadores de datos a partir de las carpetas
train_generator = train_datagen.flow_from_directory(
    '../data/raw/dataset/train',
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    '../data/raw/dataset/test',
    target_size=(150, 150),
    batch_size=32,
    shuffle=False
)


Found 20000 images belonging to 2 classes.
Found 5000 images belonging to 1 classes.


In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

In [19]:
from tensorflow.keras.applications import VGG16

#Cargar modelo preentrenado VGG16 sin la capa superior (clasificación)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))

#Congelar las capas del modelo base para que no se entrenen durante el ajuste fino
base_model.trainable = False

#Crear el modelo secuencial y agregar las capas del modelo base
model = Sequential()
model.add(base_model)
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dense(1, activation='sigmoid'))  # Salida binaria (gato o perro)



In [20]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(
    train_generator,
    epochs=3
)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 49s 78ms/step - accuracy: 0.9639 - loss: 0.0947
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 51s 81ms/step - accuracy: 0.9638 - loss: 0.0883
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 47s 74ms/step - accuracy: 0.9643 - loss: 0.0952
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 73ms/step - accuracy: 0.9725 - loss: 0.0710
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 74ms/step - accuracy: 0.9717 - loss: 0.0718


In [37]:
import numpy as np

predicciones = model.predict(test_generator)

predicciones


157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 68ms/step


array([[5.0086413e-10],
       [3.3496011e-07],
       [2.1120525e-06],
       ...,
       [9.9900407e-01],
       [9.8930383e-01],
       [1.6975878e-02]], dtype=float32)

In [38]:
clases_predichas = (predicciones > 0.5).astype(int).ravel()  # Convertir a 0 o 1 y a un array unidimensional

etiquetas_mapa = {v: k for k, v in train_generator.class_indices.items()}
resultados = [etiquetas_mapa[clase] for clase in clases_predichas]

for archivo, resultado in zip(test_generator.filenames[:10], resultados[:10]):
    print(f'Archivo: {archivo}, Predicción: {resultado}')

Archivo: test/cat.10.jpg, Predicción: cat
Archivo: test/cat.10001.jpg, Predicción: cat
Archivo: test/cat.10002.jpg, Predicción: cat
Archivo: test/cat.10006.jpg, Predicción: cat
Archivo: test/cat.10018.jpg, Predicción: cat
Archivo: test/cat.10024.jpg, Predicción: cat
Archivo: test/cat.10031.jpg, Predicción: cat
Archivo: test/cat.10036.jpg, Predicción: cat
Archivo: test/cat.10038.jpg, Predicción: cat
Archivo: test/cat.10043.jpg, Predicción: dog


In [ ]:
loss, accuracy = model.evaluate(test_generator)

print(f'Pérdida en el conjunto de prueba: {loss}')
print(f'Precisión en el conjunto de prueba: {accuracy}')


157/157 ━━━━━━━━━━━━━━━━━━━━ 11s 71ms/step - accuracy: 0.5104 - loss: 5.1473
Pérdida en el conjunto de prueba: 5.147332191467285
Precisión en el conjunto de prueba: 0.5103999972343445


In [41]:
model.save('../models/cats_vs_dogs_model.keras')